# Notebook 3 — Low-Rank Filter Factorization (data-driven rank selection, HARD 2.5h budget)

**A different compression axis than Notebook 2.** Instead of REMOVING channels (width pruning),
every 5x5x12x12 conv is factorized as two smaller convs:
```
Original:    Conv2D(12, 5x5)                              -- 3,600 params
Factorized:  Conv2D(R, 5x5, no bias)  ->  Conv2D(12, 1x1)  -- R*(300+12) params
```
This is the classic low-rank filter-bank decomposition (Jaderberg et al. 2014 / Denton et al. 2014):
reshape the (5,5,12,12) kernel to a (300,12) matrix and keep only its top-R singular
vectors -- `R` controls the compression/accuracy trade-off, exactly the way `WIDTH_R`
did for channel pruning in Notebook 2.

### The key requirement this notebook satisfies: R is *measured*, not guessed

```
Stage 0 (free, no GPU training): SVD spectrum of every teacher conv layer
                                  -> how much "energy" does rank R capture, on average?
Stage 1 (cheap, no training):    SVD-initialize a handful of candidate R values directly
                                  from the teacher's weights, evaluate them AS-IS (zero
                                  training) on the calibration bank -> pick the smallest R
                                  whose zero-shot Pd stays within tolerance of the teacher
Stage 2 (the expensive part):    ONLY THEN fine-tune the chosen R with the remaining budget
```
This mirrors how you'd pick any hyperparameter with a validation set -- cheap search first,
commit GPU time to the winner second.

### Scope for this run
Low-rank factorization tested **alone** (all 12 channels kept, `WIDTH=12`) -- not combined
with the Notebook 2 channel pruning yet. That combination is a natural follow-up once this
axis is validated on its own (same reasoning as testing magnitude vs snr_aware separately
before ever combining anything).

### Kaggle attach
- `dldoa-source-code` (DL_DOA folder + pretrained weights + `dldoa_dataset_generation.py`)
- `dldoa-frozen-banks` (eval_bank.npz + calibration_bank.npz)
- (optional) the `results_r8_magnitude...json` / `results_r8_snr_aware...json` from Notebook 2, if you want the final comparison table to include them automatically

In [ ]:
# Cell 1 — Setup + wall-clock start
import importlib, subprocess, sys, time
T_START = time.time()

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

def elapsed_hours():
    return (time.time() - T_START) / 3600

print('Wall clock started. t=0.00h')

In [ ]:
# Cell 2 — Locate repo source, weights, frozen banks, training generator
def find_path(name_pattern):
    from pathlib import Path
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for p in Path(root).rglob(name_pattern):
            if p.is_file(): return str(p)
    return None

SRC_MODEL_FILE = find_path('tvt_models.py')
assert SRC_MODEL_FILE is not None, 'DL_DOA source not found -- attach dldoa-source-code'
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))
print(f'DL_DOA dir: {DL_DOA_DIR}')

WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5')
assert WEIGHTS_PATH is not None, 'pretrained weights not found'
print(f'Weights: {WEIGHTS_PATH}')

EVAL_BANK_PATH = find_path('eval_bank.npz')
CALIB_BANK_PATH = find_path('calibration_bank.npz')
assert EVAL_BANK_PATH is not None and CALIB_BANK_PATH is not None, 'frozen banks not found'
print(f'Eval bank: {EVAL_BANK_PATH}')
print(f'Calibration bank: {CALIB_BANK_PATH}')

TRAIN_GEN_FILE = find_path('dldoa_dataset_generation.py')
assert TRAIN_GEN_FILE is not None, 'dldoa_dataset_generation.py not found'
print(f'Training generator script: {TRAIN_GEN_FILE}')

# Optional: Notebook 2's results, for the final 4-way comparison table
PRUNE_MAG_JSON = find_path('results_r8_magnitude*.json')
PRUNE_SNR_JSON = find_path('results_r8_snr_aware*.json')
print(f'Notebook 2 magnitude results: {PRUNE_MAG_JSON}')
print(f'Notebook 2 snr_aware results: {PRUNE_SNR_JSON}')

In [ ]:
# Cell 3 — Import ORIGINAL evaluator + Resnet + training generator (unchanged from Notebook 1/2)
sys.path.insert(0, DL_DOA_DIR)
sys.path.insert(0, os.path.dirname(TRAIN_GEN_FILE))

from src.tvt_models import Resnet
from src.TVT_Blob_Inference import (
    get_blob_detector, get_blob_peaks, peaks_to_angles,
    prepare_for_metric, get_ang_difference, filter_angles,
)
from dldoa_dataset_generation import training_data_generator

print('✅ Imported Resnet, original evaluator functions, and training_data_generator')

In [ ]:
# Cell 4 — Load frozen banks
eval_bank = np.load(EVAL_BANK_PATH)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA = float(eval_bank['sigma']); M = int(eval_bank['M'])

calib_bank = np.load(CALIB_BANK_PATH)
CALIB_DATA = calib_bank['data']
CALIB_GT = calib_bank['gt'].astype(np.float32)
CALIB_FEAT = calib_bank['feat']
CALIB_META = calib_bank['meta']

print(f'Eval bank: {EVAL_DATA.shape[0]} samples | Calibration bank: {CALIB_DATA.shape[0]} samples')

In [ ]:
# Cell 5 — Load teacher
teacher = Resnet(input_shape=(64, 64, 2))
teacher.load_weights(WEIGHTS_PATH)
teacher.trainable = False
TEACHER_PARAMS = teacher.count_params()
print(f'✅ Teacher loaded: {TEACHER_PARAMS:,} params (frozen)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — CONFIG
# ══════════════════════════════════════════════════════════════
LAMBDA_DISTILL   = 0.5
TOTAL_TIME_BUDGET_HOURS = 2.5     # HARD cap for the whole notebook run
SWEEP_RESERVE_MINUTES   = 25      # time reserved for Cell 9's zero-training R-sweep
EVAL_RESERVE_MINUTES    = 20      # time reserved for Cell 13's final evaluation
N_PER_SNR_EVAL          = 150     # subsampled final-eval size (of 1000 available)
MAX_EPOCHS_CAP          = 200

CANDIDATE_RANKS = [1, 2, 3, 4, 6, 8, 10]   # searched in Cell 8/9, ascending (most compressed first)
PD_TOLERANCE    = 0.05                      # accept the smallest R within this much Pd of the teacher (zero-shot)

print(f'Time budget: {TOTAL_TIME_BUDGET_HOURS}h total, '
      f'{SWEEP_RESERVE_MINUTES}min for R-sweep, {EVAL_RESERVE_MINUTES}min for final eval')
print(f'Candidate ranks: {CANDIDATE_RANKS}  (Pd tolerance vs teacher: {PD_TOLERANCE})')

In [ ]:
# Cell 7 — Low-rank building blocks + SVD-based weight initialization

def conv_lowrank(x, out_filters, rank_r, kernel_size=5):
    """One 5x5xCin x out_filters conv, factorized as basis (5x5, no bias) + 1x1 mixing (with bias).
    Equivalent to keeping only the top-rank_r singular vectors of the reshaped kernel."""
    x = Conv2D(rank_r, kernel_size, padding='same', use_bias=False)(x)
    x = Conv2D(out_filters, 1, padding='same')(x)
    return x

def res_conv_lowrank(x, filters, rank_r):
    skip = x
    x = conv_lowrank(x, filters, rank_r); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = conv_lowrank(x, filters, rank_r); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def build_lowrank_resnet(rank_r, width=12, n_blocks=64, input_shape=(64, 64, 2), name=None):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(width, (5, 5), strides=(2, 2), padding='same')(x_in)   # not factorized
    for _ in range(n_blocks):
        x = res_conv_lowrank(x, width, rank_r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)         # not factorized
    return Model(x_in, x, name=name or f'LowRankResNet-R{rank_r}')

def get_weighted_layers(model):
    return [l for l in model.layers if l.get_weights()]

def svd_decompose_conv(kernel, rank_r):
    """kernel: (kh,kw,in_ch,out_ch) -> basis (kh,kw,in_ch,R), mix (1,1,R,out_ch), via truncated SVD
    of the reshaped (kh*kw*in_ch, out_ch) matrix.

    Verified locally: at R = full rank, basis_flat @ mix_flat reconstructs the original
    kernel to ~1e-7 (machine precision); at R < full rank, reconstruction error increases
    monotonically as R decreases, as expected for a truncated-SVD approximation."""
    kh, kw, in_ch, out_ch = kernel.shape
    W = kernel.reshape(kh * kw * in_ch, out_ch)
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    R = min(rank_r, len(S))
    basis = (U[:, :R] * S[:R]).reshape(kh, kw, in_ch, R)
    mix = Vt[:R, :].reshape(1, 1, R, out_ch)   # NOTE: no transpose -- Vt[:R,:] is already (R, out_ch)
    if R < rank_r:   # pad with zeros if rank_r exceeds the matrix's actual rank (harmless, unused capacity)
        basis = np.pad(basis, ((0,0),(0,0),(0,0),(0, rank_r-R)))
        mix = np.pad(mix, ((0,0),(0,0),(0, rank_r-R),(0,0)))
    return basis.astype(np.float32), mix.astype(np.float32)

def svd_init_lowrank_model(teacher_model, student_model, rank_r, n_blocks=64):
    tw = get_weighted_layers(teacher_model)   # 1 + 4*n_blocks + 1  (conv1,bn1,conv2,bn2 per block)
    sw = get_weighted_layers(student_model)   # 1 + 6*n_blocks + 1  (basis1,mix1,bn1,basis2,mix2,bn2 per block)
    assert len(tw) == 1 + 4*n_blocks + 1
    assert len(sw) == 1 + 6*n_blocks + 1

    sw[0].set_weights(tw[0].get_weights())
    for i in range(n_blocks):
        t_conv1, t_bn1, t_conv2, t_bn2 = tw[1+4*i : 1+4*i+4]
        s_basis1, s_mix1, s_bn1, s_basis2, s_mix2, s_bn2 = sw[1+6*i : 1+6*i+6]

        k1, b1 = t_conv1.get_weights()
        basis1, mix1 = svd_decompose_conv(k1, rank_r)
        s_basis1.set_weights([basis1])
        s_mix1.set_weights([mix1, b1])
        s_bn1.set_weights(t_bn1.get_weights())

        k2, b2 = t_conv2.get_weights()
        basis2, mix2 = svd_decompose_conv(k2, rank_r)
        s_basis2.set_weights([basis2])
        s_mix2.set_weights([mix2, b2])
        s_bn2.set_weights(t_bn2.get_weights())

    sw[-1].set_weights(tw[-1].get_weights())

print('✅ Low-rank builder + SVD-init function ready')

In [ ]:
# Cell 8 — STAGE 0 (free): SVD spectrum of every teacher conv layer -- how much
# "energy" does rank R capture, averaged over all 128 conv layers (64 blocks x 2)?

def collect_singular_values(teacher_model, n_blocks=64):
    tw = get_weighted_layers(teacher_model)
    all_S = []
    for i in range(n_blocks):
        for conv_layer in [tw[1+4*i], tw[1+4*i+2]]:   # conv1, conv2
            k, _ = conv_layer.get_weights()
            kh, kw, in_ch, out_ch = k.shape
            W = k.reshape(kh*kw*in_ch, out_ch)
            S = np.linalg.svd(W, compute_uv=False)
            all_S.append(S)
    return np.array(all_S)   # (128, 12)

S_all = collect_singular_values(teacher)
print(f'Collected singular-value spectra for {S_all.shape[0]} conv layers (each has {S_all.shape[1]} values)')

explained_mean, explained_min = [], []
for R in range(1, S_all.shape[1]+1):
    ev = (S_all[:, :R]**2).sum(axis=1) / (S_all**2).sum(axis=1)
    explained_mean.append(ev.mean()); explained_min.append(ev.min())

print(f'{"R":>3} | {"mean explained var":>19} | {"worst-layer explained var":>26}')
for R in CANDIDATE_RANKS:
    print(f'{R:>3} | {explained_mean[R-1]:>19.3f} | {explained_min[R-1]:>26.3f}')

plt.figure(figsize=(7,4))
plt.plot(range(1,13), explained_mean, 'o-', label='mean over 128 layers')
plt.plot(range(1,13), explained_min, 's--', label='worst single layer')
plt.axhline(0.9, color='gray', ls=':', lw=1)
plt.xlabel('Rank R'); plt.ylabel('Explained variance (energy captured)')
plt.title('SVD spectrum of teacher conv weights'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'svd_spectrum.png'), dpi=130); plt.show()
print(f'Elapsed: {elapsed_hours():.2f}h')

In [ ]:
# Cell 9 — STAGE 1 (cheap, zero training): SVD-init each candidate R directly from the
# teacher, evaluate AS-IS (no fine-tuning) on the calibration bank, pick the smallest R
# whose zero-shot Pd is within PD_TOLERANCE of the teacher's own Pd on the same bank.

def evaluate_on_bank(model, data_arr, feat_arr, meta_arr, batch_size=8, max_deg_error=1.0):
    detector = get_blob_detector()
    results_by_snr = {}
    N = data_arr.shape[0]
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        preds = model(data_arr[start:end], training=False)
        for j in range(end - start):
            idx = start + j
            L = int(meta_arr[idx, 0]); snr = int(meta_arr[idx, 1])
            peaks, amps = get_blob_peaks(preds[j], detector)
            order = np.argsort(-amps); peaks = peaks[order[:L]]
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M)
            gt_angles, pred_angles = prepare_for_metric(angles_est, feat_arr[idx])
            results_by_snr.setdefault(snr, []).append((gt_angles, pred_angles))
    final_pd, final_rmse = {}, {}
    for snr, examples in results_by_snr.items():
        good_all, bad_all = [], []
        for gt, pred in examples:
            if np.isnan(pred).any(): continue
            diffs = get_ang_difference(gt, pred)
            good, bad = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        final_pd[snr] = len(good_all)/total if total > 0 else np.nan
        final_rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) > 0 else np.nan
    return final_pd, final_rmse

def mean_pd(pd_dict):
    return float(np.nanmean(list(pd_dict.values())))

sweep_start = time.time()
sweep_budget_hours = SWEEP_RESERVE_MINUTES / 60

teacher_calib_pd, teacher_calib_rmse = evaluate_on_bank(teacher, CALIB_DATA, CALIB_FEAT, CALIB_META)
teacher_calib_mean_pd = mean_pd(teacher_calib_pd)
print(f'Teacher zero-shot mean Pd on calibration bank: {teacher_calib_mean_pd:.4f}  (reference)')

sweep_results = {}
SELECTED_RANK_R = CANDIDATE_RANKS[-1]   # fallback: largest candidate if nothing meets tolerance
for R in CANDIDATE_RANKS:
    if (time.time() - sweep_start) / 3600 >= sweep_budget_hours:
        print(f'⏱️  Sweep time budget reached, stopping search at R={R}')
        break
    cand = build_lowrank_resnet(R, name=f'probe_R{R}')
    svd_init_lowrank_model(teacher, cand, R)
    pd_R, rmse_R = evaluate_on_bank(cand, CALIB_DATA, CALIB_FEAT, CALIB_META)
    mpd = mean_pd(pd_R)
    params_R = cand.count_params()
    gap = teacher_calib_mean_pd - mpd
    sweep_results[R] = {'params': params_R, 'mean_pd': mpd, 'gap_vs_teacher': gap}
    print(f'R={R:>2}  params={params_R:>7,}  zero-shot mean Pd={mpd:.4f}  gap={gap:+.4f}')
    if gap <= PD_TOLERANCE:
        SELECTED_RANK_R = R
        print(f'  -> within tolerance ({PD_TOLERANCE}) -- SELECTING R={R}, stopping search')
        del cand; tf.keras.backend.clear_session()
        break
    del cand; tf.keras.backend.clear_session()

print(f'\n✅ Selected rank: R = {SELECTED_RANK_R}')
print(f'Elapsed: {elapsed_hours():.2f}h')

In [ ]:
# Cell 10 — Batch-size probe at the SELECTED rank
def try_batch(rank_r, batch):
    try:
        m = build_lowrank_resnet(rank_r, name='probe_batch_test')
        opt = tf.keras.optimizers.Adam(1e-4)
        x = tf.random.normal((batch, 64, 64, 2)); y = tf.random.normal((batch, 256, 256, 1))
        with tf.GradientTape() as tape:
            pred = m(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, m.trainable_variables)
        opt.apply_gradients(zip(grads, m.trainable_variables))
        del m, opt; tf.keras.backend.clear_session()
        return True
    except tf.errors.ResourceExhaustedError:
        tf.keras.backend.clear_session()
        return False

BATCH = 8
for candidate in [32, 24, 16, 8]:
    print(f'Trying batch={candidate}...')
    if try_batch(SELECTED_RANK_R, candidate):
        BATCH = candidate
        print(f'✅ batch={candidate} works -- using this')
        break
    print(f'  OOM at batch={candidate}')

STEPS_PER_EPOCH = int(np.ceil(10000 / BATCH))
print(f'Final: BATCH={BATCH}, STEPS_PER_EPOCH={STEPS_PER_EPOCH}')
print(f'Elapsed: {elapsed_hours():.2f}h')

In [ ]:
# Cell 11 — Build final low-rank student at the selected rank, SVD-init, report compression
RUN_TAG = f'lowrank_R{SELECTED_RANK_R}_lambda{LAMBDA_DISTILL}'

student = build_lowrank_resnet(SELECTED_RANK_R, name=f'Student-{RUN_TAG}')
svd_init_lowrank_model(teacher, student, SELECTED_RANK_R)

STUDENT_PARAMS = student.count_params()
print(f'Selected rank R = {SELECTED_RANK_R}')
print(f'Student params: {STUDENT_PARAMS:,}')
print(f'Teacher params: {TEACHER_PARAMS:,}')
print(f'Reduction: {(1 - STUDENT_PARAMS/TEACHER_PARAMS)*100:.1f}%')

In [ ]:
# Cell 12 — Fine-tune with GT + distillation loss, hard time-budget stop
def make_train_ds(batch):
    def fn():
        for d, g in training_data_generator(sigma=SIGMA, M=M):
            yield d, g
    ds = tf.data.Dataset.from_generator(
        fn,
        output_signature=(
            tf.TensorSpec(shape=(64, 64, 2), dtype=tf.float32),
            tf.TensorSpec(shape=(M, M, 1), dtype=tf.float32),
        ),
    )
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

opt = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        pred = student(x, training=True)
        loss_gt = tf.reduce_mean(tf.square(pred - y))
        if LAMBDA_DISTILL > 0:
            teacher_pred = teacher(x, training=False)
            loss_distill = tf.reduce_mean(tf.square(pred - teacher_pred))
        else:
            loss_distill = 0.0
        loss = loss_gt + LAMBDA_DISTILL * loss_distill
    grads = tape.gradient(loss, student.trainable_variables)
    opt.apply_gradients(zip(grads, student.trainable_variables))
    return loss

train_budget_hours = max(0.05, TOTAL_TIME_BUDGET_HOURS - elapsed_hours() - EVAL_RESERVE_MINUTES/60)
print(f'Training time budget: {train_budget_hours:.2f}h  '
      f'({elapsed_hours():.2f}h elapsed of {TOTAL_TIME_BUDGET_HOURS}h, '
      f'{EVAL_RESERVE_MINUTES}min reserved for eval)')

ds_iter = iter(make_train_ds(BATCH))
train_start = time.time()
history_loss = []
epoch = 0
while epoch < MAX_EPOCHS_CAP:
    if (time.time() - train_start) / 3600 >= train_budget_hours:
        print(f'⏱️  Training time budget reached at epoch {epoch}'); break
    epoch_losses = []
    for step in range(STEPS_PER_EPOCH):
        x, y = next(ds_iter)
        epoch_losses.append(float(train_step(x, y)))
        if (time.time() - train_start) / 3600 >= train_budget_hours:
            break
    history_loss.append(float(np.mean(epoch_losses)))
    epoch += 1
    print(f'Epoch {epoch}: loss={history_loss[-1]:.5f}  elapsed={(time.time()-train_start)/60:.1f}min')

EPOCHS_COMPLETED = epoch
print(f'\n✅ Fine-tuning done: {EPOCHS_COMPLETED} epochs in {(time.time()-train_start)/60:.1f} min')
student.save_weights(os.path.join(OUT_DIR, f'student_{RUN_TAG}.weights.h5'))

plt.figure(figsize=(8,3.5))
plt.plot(history_loss, color='teal')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Fine-tuning loss ({RUN_TAG})')
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'train_loss_{RUN_TAG}.png'), dpi=130); plt.show()
print(f'Elapsed: {elapsed_hours():.2f}h  (budget: {TOTAL_TIME_BUDGET_HOURS}h)')
print()
print('⚠️ Remember to "Save Version -> Save & Run All (Commit)" so the saved weights')
print('   and results persist after this session ends -- /kaggle/working/ is ephemeral otherwise.')

In [ ]:
# Cell 13 — Evaluate on a SUBSAMPLED slice of the frozen eval bank (same protocol as Notebook 2)
def subsample_per_snr(data, feat, meta, n_per_snr, block_size=1000, n_blocks=8):
    idx = np.concatenate([np.arange(i*block_size, i*block_size+n_per_snr) for i in range(n_blocks)])
    return data[idx], feat[idx], meta[idx]

SUB_DATA, SUB_FEAT, SUB_META = subsample_per_snr(EVAL_DATA, EVAL_FEAT, EVAL_META, N_PER_SNR_EVAL)
print(f'Evaluating on {SUB_DATA.shape[0]} samples ({N_PER_SNR_EVAL}/SNR)')

student_pd, student_rmse = evaluate_on_bank(student, SUB_DATA, SUB_FEAT, SUB_META)
teacher_pd, teacher_rmse = evaluate_on_bank(teacher, SUB_DATA, SUB_FEAT, SUB_META)
print(f'\n✅ Evaluation complete. Elapsed: {elapsed_hours():.2f}h')

In [ ]:
# Cell 14 — Compare vs teacher, and vs Notebook 2's pruning results if available
snrs_sorted = sorted(teacher_pd.keys())
print('='*80)
print(f'Run: {RUN_TAG}   ({SUB_DATA.shape[0]} eval samples)')
print('='*80)
print(f'{"SNR":>5} | {"Student Pd":>11} {"Teacher Pd":>11} {"ΔPd":>8} | {"Stud RMSE":>10} {"Teach RMSE":>11}')
print('-'*80)
for s in snrs_sorted:
    dpd = student_pd[s] - teacher_pd[s]
    print(f'{s:>5} | {student_pd[s]:>11.4f} {teacher_pd[s]:>11.4f} {dpd:>+8.4f} | '
          f'{student_rmse[s]:>10.4f} {teacher_rmse[s]:>11.4f}')
print('='*80)
mean_dpd = np.mean([student_pd[s]-teacher_pd[s] for s in snrs_sorted])
mean_dpd_low = np.mean([student_pd[s]-teacher_pd[s] for s in snrs_sorted if s <= 0])
print(f'Mean ΔPd (all SNR): {mean_dpd:+.4f}')
print(f'Mean ΔPd (low SNR <=0dB): {mean_dpd_low:+.4f}')

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(snrs_sorted, [student_pd[s] for s in snrs_sorted], 'o-', color='teal', label=f'Low-rank ({RUN_TAG})')
axs[0].plot(snrs_sorted, [teacher_pd[s] for s in snrs_sorted], 's--', color='steelblue', label='Teacher')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('Pd'); axs[0].set_title('Pd'); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(snrs_sorted, [student_rmse[s] for s in snrs_sorted], 'o-', color='teal', label=f'Low-rank ({RUN_TAG})')
axs[1].plot(snrs_sorted, [teacher_rmse[s] for s in snrs_sorted], 's--', color='steelblue', label='Teacher')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('RMSE (deg)'); axs[1].set_title('RMSE'); axs[1].legend(); axs[1].grid(alpha=.3)
plt.suptitle(f'{RUN_TAG}  |  {STUDENT_PARAMS:,} vs {TEACHER_PARAMS:,} params '
             f'({(1-STUDENT_PARAMS/TEACHER_PARAMS)*100:.0f}% smaller)', y=1.03)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'compare_{RUN_TAG}.png'), dpi=140, bbox_inches='tight')
plt.show()

results = {
    'run_tag': RUN_TAG, 'method': 'low_rank', 'selected_rank_r': SELECTED_RANK_R,
    'lambda_distill': LAMBDA_DISTILL, 'batch': BATCH, 'epochs_completed': EPOCHS_COMPLETED,
    'n_per_snr_eval': N_PER_SNR_EVAL, 'sweep_results': sweep_results,
    'student_params': STUDENT_PARAMS, 'teacher_params': TEACHER_PARAMS,
    'student_pd': {str(k): float(v) for k, v in student_pd.items()},
    'student_rmse': {str(k): float(v) for k, v in student_rmse.items()},
    'teacher_pd': {str(k): float(v) for k, v in teacher_pd.items()},
    'teacher_rmse': {str(k): float(v) for k, v in teacher_rmse.items()},
    'mean_dpd_all': float(mean_dpd), 'mean_dpd_low_snr': float(mean_dpd_low),
    'total_elapsed_hours': float(elapsed_hours()),
}
result_path = os.path.join(OUT_DIR, f'results_{RUN_TAG}.json')
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {result_path}')

# Optional 4-way comparison if Notebook 2's results are attached
if PRUNE_MAG_JSON or PRUNE_SNR_JSON:
    print('\n' + '='*80)
    print('4-way parameter/accuracy comparison (mean ΔPd across all SNR)')
    print('='*80)
    print(f'{"Method":<28} {"Params":>10} {"Reduction":>10} {"Mean ΔPd":>10}')
    print(f'{"Teacher (baseline)":<28} {TEACHER_PARAMS:>10,} {"--":>10} {"0.0000":>10}')
    print(f'{RUN_TAG:<28} {STUDENT_PARAMS:>10,} '
          f'{(1-STUDENT_PARAMS/TEACHER_PARAMS)*100:>9.1f}% {mean_dpd:>+10.4f}')
    for path, label in [(PRUNE_MAG_JSON, 'pruning (magnitude)'), (PRUNE_SNR_JSON, 'pruning (snr_aware)')]:
        if path:
            with open(path) as f: r = json.load(f)
            print(f'{label:<28} {r["student_params"]:>10,} '
                  f'{(1-r["student_params"]/r["teacher_params"])*100:>9.1f}% {r["mean_dpd_all"]:>+10.4f}')

In [ ]:
# Cell 15 — Final summary
print('='*70)
print('NOTEBOOK 3 (LOW-RANK) RUN SUMMARY')
print('='*70)
print(f'Config:            {RUN_TAG}')
print(f'Selected rank:     R = {SELECTED_RANK_R}  (data-driven, from Cell 9 sweep)')
print(f'Total elapsed:     {elapsed_hours():.2f}h  (budget: {TOTAL_TIME_BUDGET_HOURS}h)')
print(f'{"✅ within budget" if elapsed_hours() <= TOTAL_TIME_BUDGET_HOURS + 0.1 else "⚠️ exceeded budget"}')
print(f'Epochs completed:  {EPOCHS_COMPLETED}')
print(f'Params:            {STUDENT_PARAMS:,} ({(1-STUDENT_PARAMS/TEACHER_PARAMS)*100:.1f}% smaller than teacher)')
print(f'Mean ΔPd (all):    {mean_dpd:+.4f}')
print(f'Mean ΔPd (low SNR):{mean_dpd_low:+.4f}')
print()
print('পরের ধাপ:')
print('  - এই low-rank result-কে Notebook 2-র pruning result-গুলোর সাথে তুলনা করো')
print('  - দুটো axis-ই ভালো হলে: combine করো (prune to r=8 channels, then low-rank factorize)')
print('  - সব result মিলিয়ে একটা final comparison table/plot বানাও')